In [1]:
from google.colab import drive

drive.mount("/content/gdrive")

Mounted at /content/gdrive


FUSION-01 — Initialize

In [2]:
import os
import json
import pandas as pd
import numpy as np

PROJECT_ROOT = "/content/gdrive/MyDrive/CrisisWeave_Project"

FUSION_RESULTS_DIR = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Fusion"
)

os.makedirs(FUSION_RESULTS_DIR, exist_ok=True)

print("CrisisWeave Conflict-Aware Evidence Fusion initialized.")
print("Output directory:", FUSION_RESULTS_DIR)

CrisisWeave Conflict-Aware Evidence Fusion initialized.
Output directory: /content/gdrive/MyDrive/CrisisWeave_Project/04_Results/Fusion


FUSION-02 — Load all required outputs

In [3]:
# Load unified evidence

evidence_path = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Evidence",
    "CrisisWeave_Evidence_CW001_EV001.json"
)

with open(evidence_path, "r", encoding="utf-8") as f:
    evidence_record = json.load(f)


# Load conflict assessment

conflict_path = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Conflict",
    "CrisisWeave_Conflict_Assessment_CW001_EV001.json"
)

with open(conflict_path, "r", encoding="utf-8") as f:
    conflict_assessment = json.load(f)


# Load reliability assessment

reliability_path = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Reliability",
    "CrisisWeave_Reliability_CW001_EV001.json"
)

with open(reliability_path, "r", encoding="utf-8") as f:
    reliability_summary = json.load(f)


print("Evidence:", evidence_record["evidence_id"])
print("Conflict:", conflict_assessment["evidence_id"])
print("Reliability:", reliability_summary["evidence_id"])

Evidence: EV-001
Conflict: EV-001
Reliability: EV-001


FUSION-03 — Extract source reliability

In [4]:
source_weights = {
    item["source"]: float(item["final_reliability"])
    for item in reliability_summary["sources"]
}

print("Source reliability weights:")
print(json.dumps(
    source_weights,
    indent=2
))

Source reliability weights:
{
  "CNN": 0.75,
  "BERT": 0.75,
  "OpenStreetMap": 0.9,
  "Temporal_Analysis": 0.8,
  "Response_Event": 0.85
}


FUSION-04 — Define severity evidence

In [5]:
severity_order = {
    "Low": 1,
    "Moderate": 2,
    "High": 3
}


cnn_severity = evidence_record[
    "image_evidence"
]["cnn_severity"]

temporal_severity = evidence_record[
    "temporal_evidence"
].get("latest_severity")

temporal_trend = evidence_record[
    "temporal_evidence"
]["overall_temporal_trend"]

print("CNN severity:", cnn_severity)
print("Temporal severity:", temporal_severity)
print("Temporal trend:", temporal_trend)

CNN severity: High
Temporal severity: High
Temporal trend: INCREASING


FUSION-05 — Convert severity claims into evidence distributions

In [6]:
def severity_distribution(severity):
    """
    Convert a categorical severity claim
    into a one-hot evidence distribution.
    """

    distribution = {
        "Low": 0.0,
        "Moderate": 0.0,
        "High": 0.0
    }

    if severity in distribution:
        distribution[severity] = 1.0

    return distribution


cnn_distribution = severity_distribution(
    cnn_severity
)

temporal_distribution = severity_distribution(
    temporal_severity
)

print("CNN distribution:")
print(cnn_distribution)

print("\nTemporal distribution:")
print(temporal_distribution)

CNN distribution:
{'Low': 0.0, 'Moderate': 0.0, 'High': 1.0}

Temporal distribution:
{'Low': 0.0, 'Moderate': 0.0, 'High': 1.0}


FUSION-06 — Conflict-aware severity fusion

In [7]:
def fuse_severity_evidence(
    cnn_distribution,
    temporal_distribution,
    cnn_weight,
    temporal_weight
):

    classes = [
        "Low",
        "Moderate",
        "High"
    ]

    weighted_scores = {
        c: (
            cnn_distribution[c] * cnn_weight
            +
            temporal_distribution[c] * temporal_weight
        )
        for c in classes
    }

    total_weight = (
        cnn_weight +
        temporal_weight
    )

    fused_scores = {
        c: round(
            weighted_scores[c] / total_weight,
            4
        )
        for c in classes
    }

    final_severity = max(
        fused_scores,
        key=fused_scores.get
    )

    return fused_scores, final_severity


cnn_weight = source_weights.get("CNN", 0.0)
temporal_weight = source_weights.get(
    "Temporal_Analysis",
    0.0
)

severity_scores, fused_severity = (
    fuse_severity_evidence(
        cnn_distribution,
        temporal_distribution,
        cnn_weight,
        temporal_weight
    )
)

print("Fused severity scores:")
print(json.dumps(
    severity_scores,
    indent=2
))

print("\nFused severity:", fused_severity)

Fused severity scores:
{
  "Low": 0.0,
  "Moderate": 0.0,
  "High": 1.0
}

Fused severity: High


FUSION-07 — Calculate conflict-aware uncertainty

In [8]:
def calculate_entropy(distribution):
    probabilities = np.array(
        list(distribution.values()),
        dtype=float
    )

    probabilities = probabilities[
        probabilities > 0
    ]

    if len(probabilities) == 0:
        return 1.0

    entropy = -np.sum(
        probabilities * np.log2(probabilities)
    )

    max_entropy = np.log2(3)

    return round(
        entropy / max_entropy,
        4
    )


severity_uncertainty = calculate_entropy(
    severity_scores
)

print("Normalized severity uncertainty:",
      severity_uncertainty)

Normalized severity uncertainty: -0.0


FUSION-08 — Fuse humanitarian evidence

In [9]:
bert_category = evidence_record[
    "text_evidence"
]["bert_category"]

humanitarian_support_categories = {
    "requests_or_urgent_needs",
    "displaced_people_and_evacuations",
    "injured_or_dead_people",
    "missing_or_found_people",
    "infrastructure_and_utility_damage"
}

if bert_category in humanitarian_support_categories:
    humanitarian_signal = 1.0
elif bert_category == "not_humanitarian":
    humanitarian_signal = 0.0
else:
    humanitarian_signal = 0.5

print("BERT category:", bert_category)
print("Humanitarian support:", humanitarian_signal)

BERT category: requests_or_urgent_needs
Humanitarian support: 1.0


FUSION-09 — Fuse temporal escalation

In [10]:
trend = evidence_record[
    "temporal_evidence"
]["overall_temporal_trend"]

trend_scores = {
    "INCREASING": 1.0,
    "STABLE": 0.5,
    "DECREASING": 0.0
}

temporal_escalation = trend_scores.get(
    trend,
    0.5
)

print("Temporal trend:", trend)
print("Temporal escalation score:",
      temporal_escalation)

Temporal trend: INCREASING
Temporal escalation score: 1.0


FUSION-10 — GIS critical-context signal

In [11]:
gis_evidence = evidence_record[
    "gis_evidence"
]

critical_features = 0

for feature in [
    "nearest_school",
    "nearest_hospital",
    "nearest_bridge"
]:
    value = gis_evidence.get(feature)

    if value not in [None, "", "None"]:
        critical_features += 1

gis_context_score = min(
    critical_features / 3,
    1.0
)

print("Critical GIS features:", critical_features)
print("GIS context score:", gis_context_score)

Critical GIS features: 3
GIS context score: 1.0


FUSION-11 — Response signal

In [12]:
response_status = evidence_record[
    "response_evidence"
]["response_status"]

response_scores = {
    "Dispatched": 0.5,
    "En Route": 0.75,
    "Arrived": 1.0,
    "Completed": 0.25
}

response_signal = response_scores.get(
    response_status,
    0.5
)

print("Response status:", response_status)
print("Response signal:", response_signal)

Response status: En Route
Response signal: 0.75


FUSION-12 — Build the fused dynamic state

In [13]:
fused_state = {
    "incident_id": evidence_record["incident_id"],
    "evidence_id": evidence_record["evidence_id"],

    "severity": {
        "fused_class": fused_severity,
        "scores": severity_scores,
        "uncertainty": severity_uncertainty
    },

    "humanitarian": {
        "bert_category": bert_category,
        "support_score": humanitarian_signal
    },

    "temporal": {
        "trend": trend,
        "escalation_score": temporal_escalation,
        "progression_direction":
            evidence_record[
                "temporal_evidence"
            ]["progression_direction"]
    },

    "gis": {
        "critical_context_score":
            round(gis_context_score, 4)
    },

    "response": {
        "status": response_status,
        "response_signal":
            response_signal
    },

    "conflict": {
        "status":
            conflict_assessment[
                "severity_conflict"
            ]["status"],
        "overall_conflict":
            conflict_assessment[
                "overall_conflict"
            ]
    }
}

print(json.dumps(
    fused_state,
    indent=2,
    ensure_ascii=False
))

{
  "incident_id": "CW-001",
  "evidence_id": "EV-001",
  "severity": {
    "fused_class": "High",
    "scores": {
      "Low": 0.0,
      "Moderate": 0.0,
      "High": 1.0
    },
    "uncertainty": -0.0
  },
  "humanitarian": {
    "bert_category": "requests_or_urgent_needs",
    "support_score": 1.0
  },
  "temporal": {
    "trend": "INCREASING",
    "escalation_score": 1.0,
    "progression_direction": "NE"
  },
  "gis": {
    "critical_context_score": 1.0
  },
  "response": {
    "status": "En Route",
    "response_signal": 0.75
  },
  "conflict": {
    "status": "AGREEMENT",
    "overall_conflict": false
  }
}


FUSION-13 — Save fused state

In [14]:
fusion_json_path = os.path.join(
    FUSION_RESULTS_DIR,
    "CrisisWeave_Fused_State_CW001_EV001.json"
)

with open(
    fusion_json_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        fused_state,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Fused dynamic state saved:")
print(fusion_json_path)

Fused dynamic state saved:
/content/gdrive/MyDrive/CrisisWeave_Project/04_Results/Fusion/CrisisWeave_Fused_State_CW001_EV001.json


FUSION-14 — Controlled conflict scenario

In [15]:
# FUSION-14 — Controlled conflicting-evidence test

# Scenario:
# CNN says HIGH
# Temporal evidence says LOW
#
# This is intentionally created only to test
# the conflict-aware fusion mechanism.

test_cnn_distribution = severity_distribution("High")
test_temporal_distribution = severity_distribution("Low")

test_cnn_weight = source_weights.get("CNN", 0.75)
test_temporal_weight = source_weights.get(
    "Temporal_Analysis",
    0.80
)

test_severity_scores, test_fused_severity = (
    fuse_severity_evidence(
        test_cnn_distribution,
        test_temporal_distribution,
        test_cnn_weight,
        test_temporal_weight
    )
)

test_uncertainty = calculate_entropy(
    test_severity_scores
)

print("CONTROLLED CONFLICT SCENARIO")
print("----------------------------")
print("CNN severity      : High")
print("Temporal severity : Low")
print()
print("Fused scores:")
print(json.dumps(
    test_severity_scores,
    indent=2
))
print()
print("Fused severity    :", test_fused_severity)
print("Uncertainty       :", test_uncertainty)

CONTROLLED CONFLICT SCENARIO
----------------------------
CNN severity      : High
Temporal severity : Low

Fused scores:
{
  "Low": 0.5161,
  "Moderate": 0.0,
  "High": 0.4839
}

Fused severity    : Low
Uncertainty       : 0.6305


FUSION-15 — Compare agreement vs conflict

In [16]:
# FUSION-15 — Compare agreement and conflict scenarios

comparison = pd.DataFrame([
    {
        "scenario": "Agreement",
        "CNN": "High",
        "Temporal": "High",
        "fused_severity": fused_severity,
        "uncertainty": severity_uncertainty
    },
    {
        "scenario": "Conflict",
        "CNN": "High",
        "Temporal": "Low",
        "fused_severity": test_fused_severity,
        "uncertainty": test_uncertainty
    }
])

display(comparison)

,scenario,CNN,Temporal,fused_severity,uncertainty
0,Agreement,High,High,High,-0.0000
1,Conflict,High,Low,Low,0.6305


FUSION-16 — Build conflict-aware test state

In [17]:
# FUSION-16 — Store controlled conflict experiment

conflict_test_state = {
    "scenario": "CONTROLLED_CONFLICT_TEST",

    "inputs": {
        "cnn_severity": "High",
        "temporal_severity": "Low"
    },

    "source_weights": {
        "CNN": test_cnn_weight,
        "Temporal_Analysis": test_temporal_weight
    },

    "fusion_result": {
        "severity_scores": test_severity_scores,
        "fused_severity": test_fused_severity,
        "uncertainty": test_uncertainty
    },

    "purpose": (
        "Demonstrate behavior of conflict-aware "
        "fusion under contradictory severity evidence."
    )
}

print(json.dumps(
    conflict_test_state,
    indent=2
))

{
  "scenario": "CONTROLLED_CONFLICT_TEST",
  "inputs": {
    "cnn_severity": "High",
    "temporal_severity": "Low"
  },
  "source_weights": {
    "CNN": 0.75,
    "Temporal_Analysis": 0.8
  },
  "fusion_result": {
    "severity_scores": {
      "Low": 0.5161,
      "Moderate": 0.0,
      "High": 0.4839
    },
    "fused_severity": "Low",
    "uncertainty": 0.6305
  },
  "purpose": "Demonstrate behavior of conflict-aware fusion under contradictory severity evidence."
}


FUSION-17 — Save the controlled experiment

In [18]:
# FUSION-17 — Save controlled conflict experiment

conflict_test_path = os.path.join(
    FUSION_RESULTS_DIR,
    "CrisisWeave_Controlled_Conflict_Test.json"
)

with open(
    conflict_test_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        conflict_test_state,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Controlled conflict experiment saved:")
print(conflict_test_path)

Controlled conflict experiment saved:
/content/gdrive/MyDrive/CrisisWeave_Project/04_Results/Fusion/CrisisWeave_Controlled_Conflict_Test.json
